In [3]:
import time
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("garments_worker_productivity.csv")
print(df.shape)
print(df.isna().sum())

(1197, 15)
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


In [5]:
df["department"] = df["department"].str.strip()
df["department"] = df["department"].replace("sweing", "sewing")

df = df.drop(columns=["date"])

df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)

print(df["department"].unique())
print(df["MeetsTarget"].value_counts())

['sewing' 'finishing']
MeetsTarget
1    875
0    322
Name: count, dtype: int64


In [6]:
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_df = train_df.sort_index()
test_df = test_df.sort_index()

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("train index head:", train_df.index[:10].tolist())
print("test index head:", test_df.index[:10].tolist())

train shape: (958, 15)
test shape: (239, 15)
train index head: [0, 2, 3, 4, 5, 6, 7, 9, 10, 11]
test index head: [1, 8, 13, 14, 20, 21, 34, 36, 40, 53]


In [7]:
wip_median = train_df["wip"].median()
print("wip median from train:", wip_median)

train_df = train_df.copy()
test_df = test_df.copy()

train_df["wip"] = train_df["wip"].fillna(wip_median)
test_df["wip"] = test_df["wip"].fillna(wip_median)

print("remaining NaN in train:", train_df["wip"].isna().sum())
print("remaining NaN in test:", test_df["wip"].isna().sum())

wip median from train: 1047.0
remaining NaN in train: 0
remaining NaN in test: 0


In [8]:
cat_cols = ["quarter", "department", "day", "team"]
num_cols = ["targeted_productivity", "smv", "wip", "over_time",
            "incentive", "idle_time", "idle_men",
            "no_of_style_change", "no_of_workers"]

train_dummies = pd.get_dummies(train_df[cat_cols], columns=cat_cols)
test_dummies = pd.get_dummies(test_df[cat_cols], columns=cat_cols)

# align columns so both have the same dummy features, in the same order
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_dummies = train_dummies.astype(float)
test_dummies = test_dummies.astype(float)

print("train dummies shape:", train_dummies.shape)
print("test dummies shape:", test_dummies.shape)
print("columns match:", list(train_dummies.columns) == list(test_dummies.columns))

train dummies shape: (958, 25)
test dummies shape: (239, 25)
columns match: True


In [9]:
num_mean = train_df[num_cols].mean()
num_std = train_df[num_cols].std(ddof=0)

train_num_scaled = (train_df[num_cols] - num_mean) / num_std
test_num_scaled = (test_df[num_cols] - num_mean) / num_std

train_X = pd.concat([train_num_scaled, train_dummies], axis=1)
test_X = pd.concat([test_num_scaled, test_dummies], axis=1)

print("train_X shape:", train_X.shape)
print("test_X shape:", test_X.shape)
print(train_X.head())

train_X shape: (958, 34)
test_X shape: (239, 34)
   targeted_productivity       smv       wip  over_time  incentive  idle_time  \
0               0.712301  1.007667  0.012427   0.744469   0.420034  -0.063208   
2               0.712301 -0.338061 -0.119475  -0.281716   0.091399  -0.063208   
3               0.712301 -0.338061 -0.119475  -0.281716   0.091399  -0.063208   
4               0.712301  0.983945  0.070841  -0.803810   0.091399  -0.063208   
5               0.712301  0.983945 -0.104400   0.636450   0.009241  -0.063208   

   idle_men  no_of_style_change  no_of_workers  quarter_Quarter1  ...  team_3  \
0 -0.111978           -0.338554       1.094043               1.0  ...     0.0   
2 -0.111978           -0.338554      -0.190965               1.0  ...     0.0   
3 -0.111978           -0.338554      -0.190965               1.0  ...     0.0   
4 -0.111978           -0.338554       0.958779               1.0  ...     0.0   
5 -0.111978           -0.338554       0.958779             

In [10]:
X_train_reg = train_X.values
y_train_reg = train_df["actual_productivity"].values

X_test_reg = test_X.values
y_test_reg = test_df["actual_productivity"].values

# add intercept column
X_train_reg_b = np.hstack([np.ones((X_train_reg.shape[0], 1)), X_train_reg])
X_test_reg_b = np.hstack([np.ones((X_test_reg.shape[0], 1)), X_test_reg])

t0 = time.perf_counter()
w_reg, _, _, _ = np.linalg.lstsq(X_train_reg_b, y_train_reg, rcond=None)
train_time_reg = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg = X_test_reg_b @ w_reg
pred_time_reg = time.perf_counter() - t0

mae = np.mean(np.abs(y_test_reg - y_pred_reg))
rmse = np.sqrt(np.mean((y_test_reg - y_pred_reg) ** 2))
ss_res = np.sum((y_test_reg - y_pred_reg) ** 2)
ss_tot = np.sum((y_test_reg - np.mean(y_test_reg)) ** 2)
r2 = 1 - ss_res / ss_tot

print("Linear Regression (manual)")
print("train time:   {:.6f} s".format(train_time_reg))
print("predict time: {:.6f} s".format(pred_time_reg))
print("MAE:  {:.6f}".format(mae))
print("RMSE: {:.6f}".format(rmse))
print("R2:   {:.6f}".format(r2))

Linear Regression (manual)
train time:   0.005915 s
predict time: 0.000064 s
MAE:  0.101798
RMSE: 0.141897
R2:   0.348388


In [11]:
X_train_clf = train_X.values
y_train_clf = train_df["MeetsTarget"].values

X_test_clf = test_X.values
y_test_clf = test_df["MeetsTarget"].values

# standardize again for logistic (same scaling is fine; already standardized)
X_train_clf_b = np.hstack([np.ones((X_train_clf.shape[0], 1)), X_train_clf])
X_test_clf_b = np.hstack([np.ones((X_test_clf.shape[0], 1)), X_test_clf])


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def logistic_fit(X, y, lr=0.1, epochs=2000, l2=0.0):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(epochs):
        z = X @ w
        p = sigmoid(z)
        grad = (X.T @ (p - y)) / n
        if l2 > 0:
            grad[1:] += (l2 / n) * w[1:]
        w -= lr * grad
    return w


t0 = time.perf_counter()
w_clf = logistic_fit(X_train_clf_b, y_train_clf, lr=0.1, epochs=2000)
train_time_clf = time.perf_counter() - t0

t0 = time.perf_counter()
probs = sigmoid(X_test_clf_b @ w_clf)
y_pred_clf = (probs >= 0.5).astype(int)
pred_time_clf = time.perf_counter() - t0

tp = np.sum((y_pred_clf == 1) & (y_test_clf == 1))
fp = np.sum((y_pred_clf == 1) & (y_test_clf == 0))
fn = np.sum((y_pred_clf == 0) & (y_test_clf == 1))
tn = np.sum((y_pred_clf == 0) & (y_test_clf == 0))

acc = (tp + tn) / len(y_test_clf)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

print("Logistic Regression (manual)")
print("train time:   {:.6f} s".format(train_time_clf))
print("predict time: {:.6f} s".format(pred_time_clf))
print("accuracy:  {:.6f}".format(acc))
print("precision: {:.6f}".format(prec))
print("recall:    {:.6f}".format(rec))
print("f1:        {:.6f}".format(f1))
print()
print("confusion: TP={} FP={} FN={} TN={}".format(tp, fp, fn, tn))

Logistic Regression (manual)
train time:   0.047721 s
predict time: 0.000054 s
accuracy:  0.757322
precision: 0.783920
recall:    0.912281
f1:        0.843243

confusion: TP=156 FP=43 FN=15 TN=25
